In [ ]:
# dbutils.widgets.text() creates a notebook parameter called catalog
# "predictivemaintenance_dev" is the default value when no value is supplied.
#dbutils.widgets.get("catalog") retrieves the selected catalog.
#CATALOG is the Python variable we'll use throughout Notebook 31.


dbutils.widgets.text(
    "catalog",
    "predictivemaintenance_dev",
    "Target Catalog"
)

CATALOG = dbutils.widgets.get("catalog")

print(f"Target catalog: {CATALOG}")

In [ ]:
# Identify the deployment environment
dbutils.widgets.text("deployment_target", "dev")
DEPLOYMENT_TARGET = dbutils.widgets.get("deployment_target")

# Expected catalog for each environment
EXPECTED_CATALOGS = {
    "dev": "predictivemaintenance_dev",
    "prod": "predictivemaintenancecatalog"
}

# Prevent accidental writes to the wrong catalog
if DEPLOYMENT_TARGET not in EXPECTED_CATALOGS:
    raise ValueError(
        f"Unknown deployment target: {DEPLOYMENT_TARGET}"
    )

if CATALOG != EXPECTED_CATALOGS[DEPLOYMENT_TARGET]:
    raise ValueError(
        f"Catalog {CATALOG} does not match "
        f"deployment target {DEPLOYMENT_TARGET}"
    )

print(f"Deployment target: {DEPLOYMENT_TARGET}")
print(f"Catalog: {CATALOG}")

## Step 1 — Read and inspect Silver

In [0]:

from pyspark.sql import functions as F
import math

silver_table = f"{CATALOG}.silver_data.streaming"
gold_table = f"{CATALOG}.gold_data.streaming"

# Create Gold schema if needed
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold_data
""")

# Reuse the checkpoint Volume
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS
{CATALOG}.bronze_data_not_quality_checked.checkpoints
""")

# Different checkpoint from Bronze -> Silver
checkpoint_path = (
    f"/Volumes/{CATALOG}/"
    "bronze_data_not_quality_checked/"
    "checkpoints/silver_to_gold"
)
print(f"Silver table: {silver_table}")
print(f"Gold table: {gold_table}")
print(f"Checkpoint: {checkpoint_path}")

## Step 2 — Read Silver as a stream

In [0]:

silver_stream = (
    spark.readStream
    .option("skipChangeCommits", "true")
    .table(silver_table)
)

print("Is streaming:", silver_stream.isStreaming)
silver_stream.printSchema()


## Step 3 — Apply feature engineering

In [0]:

# Only quality-passed records go to Gold
valid_stream = silver_stream.filter(
    F.col("_quality_pass") == True
)

# Apply the training feature calculations
gold_features = (
    valid_stream
    .withColumn(
        "Mechanical_power_W",
        F.lit(2 * math.pi / 60)
        * F.col("Rotational_speed_rpm")
        * F.col("Torque_Nm")
    )
    .withColumn(
        "Temperature_difference_K",
        F.col("Process_temperature_K")
        - F.col("Air_temperature_K")
    )
    .withColumn(
        "Torque_wear",
        F.col("Torque_Nm") * F.col("Tool_wear_min")
    )
    .withColumn(
        "Speed_wear",
        F.col("Rotational_speed_rpm")
        * F.col("Tool_wear_min")
    )
    .withColumn(
        "Power_wear",
        F.col("Mechanical_power_W")
        * F.col("Tool_wear_min")
    )
)


## Step 4 — Select our model-ready columns

In [0]:

feature_columns = [
    "Type",
    "Air_temperature_K",
    "Process_temperature_K",
    "Rotational_speed_rpm",
    "Torque_Nm",
    "Tool_wear_min",
    "Mechanical_power_W",
    "Temperature_difference_K",
    "Torque_wear",
    "Speed_wear",
    "Power_wear",
    "Machine_failure"
]

gold_stream_df = (
    gold_features
    .select(
        "UDI",
        *feature_columns,
        F.current_timestamp().alias("_gold_processed_at")
    )
)

gold_stream_df.printSchema()


In [0]:
#import time

# Streaming display isn't supported for aggregation-free streams here,
# so materialize a one-shot preview into an in-memory table first
# (a fresh checkpoint per run keeps the preview re-runnable)
#preview_query = (
    #gold_stream_df.writeStream
    #.format("memory")
    #.queryName("gold_preview")
    #.outputMode("append")
    #.option(
        #"checkpointLocation",
        #checkpoint_path + "_preview_" + time.strftime("%Y%m%d%H%M%S"),
    #)
    #.trigger(availableNow=True)
    ##.start()
#)
#preview_query.awaitTermination()

#display(spark.table("gold_preview"))

## Step 5 — Stream into Gold

In [0]:

# TRUNCATE removed the rows but kept the old schema (no Machine_failure);
# ACLs block automatic schema migration, so align the empty table's schema first
# Check whether the Gold table already exists
if spark.catalog.tableExists(gold_table):

    # For an existing table, add Machine_failure if missing
    if "Machine_failure" not in spark.table(gold_table).columns:
        spark.sql(
            f"ALTER TABLE {gold_table} "
            "ADD COLUMN Machine_failure BIGINT"
        )

query = (
    gold_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(gold_table)
)

query.awaitTermination()


## Step 6 — Verify Gold

In [0]:
gold_df = spark.table(gold_table)

print("Gold records:", gold_df.count())

display(gold_df.limit(10))
